In [12]:
import pandas as pd

file_path = 'Data/accepted_2007_to_2018Q4.csv.gz'

# We take the first 100,000 rows to start our Bootcamp—plenty for "complexity"
df = pd.read_csv(file_path, nrows=100000, low_memory=False)

# Get a bird's eye view
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")
print(df.columns.tolist())

#THE GREAT CULL, INVESTIGATIVE EDA OVER 151 COLUMNS.
#1. find the ghosts(>50% missing).
null_counts = df.isnull().mean() * 100
ghost_cols = null_counts[null_counts > 50].sort_values(ascending=False)

#2. find columns where every single row is the same(zero variance =  zero predictive power)
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]

#3. unique ID filter, find text columns where every row is different like names or ID - these aren't patterns, just labels.
id_like_cols = [col for col in df.columns if df[col].nunique() == len(df)]

#the report back
print(f'~Forensic EDA Report~')
print(f'Ghost Columns (>50% missing): {len(ghost_cols)}')
print(f'Constant Columns (Zero Variance): {len(constant_cols)}')
print(f'ID-like Columns (Unique Values): {len(id_like_cols)}')

#investigate the ghost columns further.
print(f'~Ghost Busting~')
print(ghost_cols.head(20))
print(constant_cols[:20])

#create a dummy target variable: 1 = Default, 0 = Paid.
df['target_dummy'] = df['loan_status'].apply(lambda x: 1 if x == 'Charged Off' else 0)

#check for columns with high correlation to the target variable.
numeric_df = df.select_dtypes(include=['number'])
correlations = numeric_df.corr()['target_dummy'].sort_values(ascending=False)

print(f'\n~Suspicious Correlation (Potential Leakage).~')
print(correlations.head(20))

#Drop the ghosts.
ghost_cols_list = null_counts[null_counts > 50].index.tolist()
df_clean = df.drop(columns=ghost_cols_list)

#Drop the constants.
constant_cols = [col for col in df_clean.columns if df_clean[col].nunique() <= 1]
df_clean = df_clean.drop(columns=constant_cols)

#rerun correlation check after cleaning.
numeric_df = df_clean.select_dtypes(include=['number'])
correlations = numeric_df.corr()['target_dummy'].sort_values(ascending=False)  

print("\n~The REAL Investigative Correlations~")
print(correlations.head(10))
print(correlations.tail(10))

# We drop anything that happens AFTER the loan is signed.
leakage_cols = [
    'recoveries', 'collection_recovery_fee', 'total_rec_prncp', 
    'total_rec_int', 'total_rec_late_fee', 'last_pymnt_amnt', 
    'hardship_dpd', 'hardship_amount', 'last_fico_range_high', 
    'last_fico_range_low'
]

# We also drop the ID columns we found earlier
id_cols = ['id', 'member_id', 'url']

# Create the Final Bootcamp Dataset
df_final = df_clean.drop(columns=leakage_cols + id_cols, errors='ignore')

print(f"Remaining Columns: {len(df_final.columns)}")
print("\nTop 5 remaining positive correlations with Default:")
print(df_final.select_dtypes(include=['number']).corr()['target_dummy'].sort_values(ascending=False).head(6))

print(df_final.shape)

df_final.to_csv('Data/processed_loan_data.csv', index=False)

Loaded 100000 rows and 151 columns.
['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verificat

C:\Users\alist\AppData\Local\Temp\ipykernel_20960\1118870010.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['target_dummy'] = df['loan_status'].apply(lambda x: 1 if x == 'Charged Off' else 0)



~Suspicious Correlation (Potential Leakage).~
target_dummy               1.000000
recoveries                 0.506465
collection_recovery_fee    0.503247
int_rate                   0.255128
hardship_dpd               0.205004
acc_open_past_24mths       0.138082
num_tl_op_past_12m         0.117897
open_rv_24m                0.108380
total_rec_late_fee         0.096877
inq_last_6mths             0.095829
open_il_24m                0.092681
open_il_12m                0.092674
open_acc_6m                0.090107
inq_last_12m               0.088190
dti                        0.086440
open_rv_12m                0.083110
inq_fi                     0.077057
all_util                   0.075046
dti_joint                  0.065067
hardship_amount            0.062974
Name: target_dummy, dtype: float64

~The REAL Investigative Correlations~
target_dummy               1.000000
recoveries                 0.506465
collection_recovery_fee    0.503247
int_rate                   0.255128
acc_open_past_2